# 🔁 GIADA Task 7b — Ca_HVA attivo nel circuito chiuso
Teacher NEURON autentico a un compartimento; controllo 0×; ladder di stimoli; formula, LUT-513 e physical-τ congelati. Il run non valida ancora il neurone Hay completo.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_7b'); GIADA_REPO=WORK/'giada'; TEACHER_REPO=WORK/'neuron_as_deep_net';WORK.mkdir(parents=True,exist_ok=True)
if not GIADA_REPO.is_dir(): subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
if not TEACHER_REPO.is_dir(): subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip();print({'giada_revision':REVISION})


In [ ]:
if shutil.which('nrnivmodl') is None:
    subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
    os.environ['PATH']=str(Path(sys.executable).parent)+os.pathsep+os.environ.get('PATH','')
assert shutil.which('nrnivmodl') and shutil.which('gcc'),'NEURON compiler/gcc non disponibili'
import torch
assert torch.cuda.is_available(),'La comparazione dei candidati congelati richiede CUDA.'
print({'cuda':torch.cuda.get_device_name(0),'nrnivmodl':shutil.which('nrnivmodl')})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
from src.giada_teacher import ExtractedGateFormula,compile_nmodl,ActiveClosedLoopCaHVAConfig
from src.giada_teacher.voltage_path_stress import EXPECTED_TASK5_ARCHIVE_SHA256,EXPECTED_TASK5_REPORT_SHA256,verified_task5_root
prereg=json.loads((GIADA_REPO/'experiments/teacher_cahva_active_closed_loop_preregistration_v1.json').read_text())
display({'question':prereg['question'],'episodes':prereg['input']['episode_count'],'exposure_gate':prereg['exposure_gate']})


In [ ]:
def file_sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()
INPUT_ROOT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK5_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
    candidates+=list(INPUT_ROOT.rglob('giada_primitive_scaling_laws.zip'))+list(INPUT_ROOT.rglob('archive.zip'))
    candidates+=[p.parent for p in INPUT_ROOT.rglob('final_report.json') if (p.parent/'frozen_scaling_checkpoints.pt').is_file()]
def exact(path):
    try:return file_sha(path)==EXPECTED_TASK5_ARCHIVE_SHA256 if path.is_file() else file_sha(path/'final_report.json')==EXPECTED_TASK5_REPORT_SHA256
    except Exception:return False
TASK5_SOURCE=next((p.resolve() for p in candidates if p.exists() and exact(p)),None)
assert TASK5_SOURCE is not None,'Aggiungi agli Input Kaggle giada_primitive_scaling_laws.zip esatto oppure imposta GIADA_TASK5_ARTIFACT.'
TASK5_ROOT=verified_task5_root(TASK5_SOURCE,Path('/kaggle/working/.task7b_task5_verified'))
print({'task5_source':str(TASK5_SOURCE),'exact_hash_verified':True})


In [ ]:
MOD=TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod'
formula=ExtractedGateFormula.from_mod(MOD)
mechanism_root=compile_nmodl(MOD,WORK/'compiled_mechanism')
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_cahva_active_closed_loop_microcanary')
assert not OUTPUT_DIR.exists() or not any(p.name not in {'.verified_task5'} for p in OUTPUT_DIR.iterdir()),f'Output con risultati già presenti: {OUTPUT_DIR}. Non sovrascrivere gli artefatti.'
config=ActiveClosedLoopCaHVAConfig();config.validate()
print({'mechanism_compiled':str(mechanism_root),'output':str(OUTPUT_DIR),'episodes':len(config.initial_voltage_mv)*len(config.gbar_multipliers)*len(config.protocol_names)})


## 🧪 Esecuzione protetta
La reference NEURON e i candidati GPU girano in processi separati. Un eventuale abort nativo restituisce un codice d'uscita senza spegnere il kernel. Anche un gate scientifico fallito produce report e ZIP.

In [ ]:
command=[sys.executable,'-u',str(GIADA_REPO/'scripts/run_cahva_closed_loop_microcanary.py'),'--design','task7b','--mod',str(MOD),'--task5-root',str(TASK5_ROOT),'--mechanism-root',str(mechanism_root),'--output-dir',str(OUTPUT_DIR),'--revision',str(REVISION)]
completed=subprocess.run(command,check=False)
if completed.returncode:raise RuntimeError(f'Task 7b terminata nel sottoprocesso (exit={completed.returncode}). Copia le ultime righe del log; il kernel resta attivo.')
report=json.loads((OUTPUT_DIR/'final_report.json').read_text())
pairs=report['causal_channel_exposure']
display({'valid':report['valid'],'reference_solver_calibrated':report['reference_solver_calibrated'],'active_exposure_valid':report['active_exposure_valid'],'active_pair_count':report['active_pair_count'],'active_pair_target':report['active_pair_target'],'formula_worst_voltage_rmse_mv':report['formula_worst_voltage_rmse_mv'],'zero_control_valid':report['zero_control_valid'],'candidate_occupancy_violations':report['candidate_occupancy_violations'],'max_lut_vs_physical_voltage_difference_mv':report['max_lut_vs_physical_voltage_difference_mv'],'pairs':pairs})
print('Il report completo è nel ZIP; non attribuire errori ai candidati se solver o esposizione falliscono.')


## 📦 Scarica lo ZIP
Metodo Blob/base64 già usato in questo progetto.

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_cahva_active_closed_loop_microcanary','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
